# 01 · Project Setup & Data Audit
*EV capacity planning for PowerCharge Utilities · notebook 1 of 8 · Anna Tafri*

Reads: `data/raw/ev_distribution_dataset.csv`, `geospatial_dataset.csv`, `weather_dataset.csv`  
Writes: `reports/data_quality_log.csv`

### Why this notebook exists
Before any capacity number reaches an executive, we need to know whether the data can support it.
This notebook answers four questions:

1. **What do we have?** Structure, grain, and coverage of each dataset.
2. **Is it clean?** Duplicates, missing values, invalid or implausible values.
3. **Does it connect?** Can EV demand, substations, and weather be joined reliably?
4. What does that mean for the analysis? A data-quality log with a decision for every issue, carried into Notebook 02.

> No data is modified here. Cleaning decisions are *recorded* in this notebook and *applied* in `02_data_transformation.ipynb`, so the audit trail stays readable.


## 1. Environment setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")

# Paths are relative to the project root so the repo runs anywhere
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print("Raw files:", sorted(p.name for p in RAW.glob("*.csv")))

pandas 3.0.5 | numpy 2.5.3
Raw files: ['ev_distribution_dataset.csv', 'geospatial_dataset.csv', 'weather_dataset.csv']


## 2. Load the raw data
Files are loaded as-is. Timestamps are parsed on load because every downstream join depends on them.

In [2]:
ev = pd.read_csv(RAW / "ev_distribution_dataset.csv", parse_dates=["Timestamp"])
geo = pd.read_csv(RAW / "geospatial_dataset.csv")
weather = pd.read_csv(RAW / "weather_dataset.csv", parse_dates=["Timestamp"])

datasets = {"ev_distribution": ev, "geospatial": geo, "weather": weather}
for name, df in datasets.items():
    print(f"{name:<16} {df.shape[0]:>4} rows x {df.shape[1]:>2} cols")

ev_distribution   500 rows x 10 cols
geospatial         50 rows x  3 cols
weather            97 rows x  5 cols


## 3. Structure overview
A reusable profile: type, completeness, cardinality, and an example value for every column.

In [3]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: dtype, nulls, distinct values, and a sample."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "example": df.iloc[0],
    })

profile(ev)

,dtype,non_null,null_pct,n_unique,example
Timestamp,datetime64[us],500,0.00,97,2020-01-10 18:00:00
Geographical_Area,str,500,0.00,3,Urban
Customer_Type,str,500,0.00,3,Industrial
Electricity_Consumption (kWh),float64,500,0.00,97,714.39
EV_Charging_Station_Location,str,500,0.00,97,"(64.01943603679787, -129.53196438913122)"
EV_Charging_Station_Specifications,str,500,0.00,3,Type 2
EV_Type,str,500,0.00,3,Electric Car
Charging_Habit,str,500,0.00,3,Occasional
Number_of_EVs,int64,500,0.00,9,8
Substation_ID,str,500,0.00,49,Substation_907


In [4]:
profile(geo)

,dtype,non_null,null_pct,n_unique,example
Substation_ID,str,50,0.00,49,Substation_907
Substation_Location,str,50,0.00,50,"(57.25813554648914, -70.90582041775649)"
Transmission_Line_Capacity (MW),int64,50,0.00,40,40


In [5]:
profile(weather)

,dtype,non_null,null_pct,n_unique,example
Timestamp,datetime64[us],97,0.00,97,2020-01-03 11:00:00
Temperature (°C),float64,97,0.00,97,28.87
Precipitation (mm),float64,97,0.00,97,0.31
Weather_Conditions,str,97,0.00,3,Snowy
Substation_ID,str,97,0.00,49,Substation_744


**Reading the profiles**
- No missing values in any table, so imputation is not needed.
- Coordinates are stored as text tuples, e.g. `"(64.02, -129.53)"`. They must be parsed into numeric `lat`/`lon` before any geospatial work.
- `Substation_ID` is the only key shared by all three tables, which makes it the backbone of the data model.

## 4. Duplicate analysis
The EV row count looks healthy at 500, but row count alone can be misleading.

In [6]:
dup_summary = pd.DataFrame({
    name: {
        "rows": len(df),
        "exact_duplicates": df.duplicated().sum(),
        "unique_rows": len(df.drop_duplicates()),
    }
    for name, df in datasets.items()
}).T
dup_summary["duplicate_pct"] = (dup_summary["exact_duplicates"] / dup_summary["rows"] * 100).round(1)
dup_summary

,rows,exact_duplicates,unique_rows,duplicate_pct
ev_distribution,500,403,97,80.60
geospatial,50,0,50,0.00
weather,97,0,97,0.00


In [7]:
# How often is each unique EV record repeated?
repeat_counts = ev.groupby(list(ev.columns)).size().value_counts().sort_index()
repeat_counts.rename_axis("times_repeated").rename("n_records").to_frame()

,n_records
times_repeated,
1,12
2,2
3,23
4,12
5,10
6,1
7,21
10,15
14,1


**Finding:** about 81% of EV rows are exact duplicates. There are only 97 unique charging events, repeated anywhere from 1 to 14 times.

Why it matters: if we keep duplicates, a substation that happens to have its rows copied 10 times will look 10× more loaded than it is. Every capacity and bottleneck conclusion would be distorted.

**Decision:** deduplicate on all columns in Notebook 02. The 97 unique rows are our true sample.

## 5. Referential integrity: do the tables connect?

In [8]:
geo_ids = set(geo["Substation_ID"])
checks = {
    "EV substations found in geospatial":      ev["Substation_ID"].isin(geo_ids).mean(),
    "Weather substations found in geospatial": weather["Substation_ID"].isin(geo_ids).mean(),
    "EV timestamps found in weather":          ev["Timestamp"].isin(weather["Timestamp"]).mean(),
}
pd.Series(checks, name="match_rate").map("{:.0%}".format).to_frame()

,match_rate
EV substations found in geospatial,100%
Weather substations found in geospatial,100%
EV timestamps found in weather,100%


In [9]:
# Weather has one row per timestamp. Does it point at the same substation as the EV record?
ev_u = ev.drop_duplicates()
tw = ev_u.merge(weather, on="Timestamp", suffixes=("_ev", "_wx"))
print(f"Joined rows: {len(tw)} of {len(ev_u)} unique EV records")
print(f"Same substation on both sides: {(tw['Substation_ID_ev'] == tw['Substation_ID_wx']).mean():.0%}")

Joined rows: 97 of 97 unique EV records
Same substation on both sides: 100%


In [10]:
# Primary-key check on the substation table
dupe_ids = geo[geo["Substation_ID"].duplicated(keep=False)]
dupe_ids

,Substation_ID,Substation_Location,Transmission_Line_Capacity (MW)
19,Substation_11,"(66.83643347824741, -69.3563579228404)",80
46,Substation_11,"(33.0822685351715, -113.11893127920547)",43


**Findings**
- Every EV and weather record maps to a known substation, and weather aligns 1:1 with EV on Timestamp + Substation_ID. That is a clean join.
- `Substation_11` appears twice in the geospatial table, with different locations and different capacities (80 MW vs 43 MW). `Substation_ID` is therefore not a valid primary key as delivered.

**Decision:** joining as-is would double Substation_11's EV demand. In Notebook 02 we will treat the two rows as distinct assets (`Substation_11_A` / `_B`) and allocate each EV record to the nearest one by coordinates. As a fallback, we can keep the conservative (lower) capacity, which is the safer choice for planning. The decision is documented either way.

## 6. Value ranges and business plausibility
The data is clean in a technical sense. The next question is whether the values make physical and business sense.

In [11]:
num_ev = ev_u.select_dtypes("number").describe().T
num_geo = geo.select_dtypes("number").describe().T
num_wx = weather.select_dtypes("number").describe().T
pd.concat({"ev": num_ev, "geospatial": num_geo, "weather": num_wx})

count   mean   std    min    25%  \
ev         Electricity_Consumption (kWh)    97.00 512.15 86.45 315.91 441.06   
           Number_of_EVs                    97.00   5.25  2.59   1.00   3.00   
geospatial Transmission_Line_Capacity (MW)  50.00  52.74 24.86  10.00  33.25   
weather    Temperature (°C)                 97.00  25.31  4.82  12.45  22.25   
           Precipitation (mm)               97.00   2.54  1.40   0.01   1.39   

                                              50%    75%    max  
ev         Electricity_Consumption (kWh)   511.35 563.39 718.98  
           Number_of_EVs                     5.00   8.00   9.00  
geospatial Transmission_Line_Capacity (MW)  51.50  74.75  94.00  
weather    Temperature (°C)                 25.25  28.90  34.10  
           Precipitation (mm)                2.69   3.66   4.97

In [12]:
for col in ["Geographical_Area", "Customer_Type", "EV_Charging_Station_Specifications",
            "EV_Type", "Charging_Habit"]:
    print(ev_u[col].value_counts().to_string(), "\n")

Geographical_Area
Rural       42
Urban       29
Suburban    26 

Customer_Type
Commercial     37
Industrial     31
Residential    29 

EV_Charging_Station_Specifications
Type 2     38
DC Fast    36
Type 1     23 

EV_Type
Electric Scooter    39
Electric Car        34
Electric Bike       24 

Charging_Habit
Daily         38
Occasional    31
Weekly        28 



In [13]:
# Plausibility check: snow at warm temperatures?
pd.crosstab(
    weather["Weather_Conditions"],
    pd.cut(weather["Temperature (°C)"], bins=[-50, 0, 2, 15, 25, 50],
           labels=["<0", "0-2", "2-15", "15-25", ">25"]),
    margins=True,
)

Temperature (°C),2-15,15-25,>25,All
Weather_Conditions,,,,
Clear,1,15,20,36
Rainy,1,16,22,39
Snowy,1,10,11,22
All,3,41,53,97


In [14]:
# Coordinate sanity: parse the text tuples and check where the network sits
def parse_coords(s: pd.Series) -> pd.DataFrame:
    xy = s.str.strip("()").str.split(",", expand=True).astype(float)
    return xy.set_axis(["lat", "lon"], axis=1)

coords = pd.concat({
    "substations": parse_coords(geo["Substation_Location"]),
    "charging_stations": parse_coords(ev_u["EV_Charging_Station_Location"]),
})
coords.groupby(level=0).agg(["min", "max"])

lat           lon       
                    min   max     min    max
charging_stations 17.72 81.87 -164.19 -62.44
substations       17.74 81.33 -158.57 -63.69

In [15]:
# Time coverage of the EV data
span = ev_u["Timestamp"].agg(["min", "max"])
print(f"EV data covers {span['min']:%d %b %Y} to {span['max']:%d %b %Y} "
      f"({(span['max'] - span['min']).days} days, {ev_u['Timestamp'].nunique()} distinct hours)")

EV data covers 03 Jan 2020 to 11 Feb 2020 (38 days, 97 distinct hours)


**Findings**
- Weather labels contradict temperatures. All 22 "Snowy" records are above 2 °C, and 11 of them are above 25 °C. The `Weather_Conditions` label is unreliable, while temperature and precipitation look internally consistent.
  to *Decision:* use temperature and precipitation as the weather signals and treat the condition label as low-confidence. We will flag this rather than silently "fix" it.
- Coordinates span latitude ~18° to ~82° N and longitude ~-164° to -62°. That is roughly all of North America up to the Arctic, which is not a realistic single-utility footprint. The data is synthetic, so the maps will be useful for *relative* clustering and distance, not for real-world placement.
- **Short history.** The data covers 38 days (Jan–Feb 2020) with 97 events. That is enough to *describe* patterns but not to *forecast* adoption with statistical confidence. Growth will be handled through explicit scenarios (for example 2×, 3×, 5× EV load) instead of trend extrapolation.
- **Unit mismatch.** Demand is in kWh per event and capacity is in MW. Comparing them requires an explicit conversion assumption (energy over a charging window to average kW). This is the most important modelling assumption in the project, and we will make it transparent in Notebook 04.

## 7. Data quality log
Every issue gets evidence, an impact rating, and a decision. This table is the contract between this notebook and the transformation step.

In [16]:
dq_log = pd.DataFrame([
    {"id": "DQ-01", "dataset": "ev_distribution", "issue": "Exact duplicate rows",
     "evidence": f"{ev.duplicated().sum()} of {len(ev)} rows ({ev.duplicated().mean():.0%}); {len(ev_u)} unique",
     "impact": "High", "decision": "Drop exact duplicates before any aggregation"},
    {"id": "DQ-02", "dataset": "geospatial", "issue": "Substation_ID not unique",
     "evidence": "Substation_11 has 2 rows with different location and capacity (80 / 43 MW)",
     "impact": "High", "decision": "Split into _A/_B and allocate EV records by nearest location"},
    {"id": "DQ-03", "dataset": "weather", "issue": "Weather label contradicts temperature",
     "evidence": "All 'Snowy' records are > 2 °C; 11 are > 25 °C",
     "impact": "Medium", "decision": "Use temperature/precipitation; flag label as low-confidence"},
    {"id": "DQ-04", "dataset": "all", "issue": "Coordinates stored as text tuples",
     "evidence": "e.g. '(64.02, -129.53)'",
     "impact": "Low", "decision": "Parse into numeric lat/lon columns"},
    {"id": "DQ-05", "dataset": "all", "issue": "Geographic footprint is unrealistic",
     "evidence": "Lat ~18–82°N, lon ~-164 to -62°",
     "impact": "Low", "decision": "Use maps for relative patterns only; note in report"},
    {"id": "DQ-06", "dataset": "ev vs geospatial", "issue": "Unit mismatch: kWh vs MW",
     "evidence": "Demand is energy per event; capacity is power",
     "impact": "High", "decision": "Convert with an explicit charging-window assumption plus sensitivity test"},
    {"id": "DQ-07", "dataset": "ev_distribution", "issue": "Short history, small sample",
     "evidence": "97 events over ~5.5 weeks",
     "impact": "Medium", "decision": "Scenario-based growth instead of time-series forecasting"},
])
dq_log.to_csv(REPORTS / "data_quality_log.csv", index=False)
dq_log

,id,dataset,issue,evidence,impact,decision
0,DQ-01,ev_distribution,Exact duplicate rows,403 of 500 rows (81%); 97 unique,High,Drop exact duplicates before any aggregation
1,DQ-02,geospatial,Substation_ID not unique,Substation_11 has 2 rows with different locati...,High,Split into _A/_B and allocate EV records by ne...
2,DQ-03,weather,Weather label contradicts temperature,All 'Snowy' records are > 2 °C; 11 are > 25 °C,Medium,Use temperature/precipitation; flag label as l...
3,DQ-04,all,Coordinates stored as text tuples,"e.g. '(64.02, -129.53)'",Low,Parse into numeric lat/lon columns
4,DQ-05,all,Geographic footprint is unrealistic,"Lat ~18–82°N, lon ~-164 to -62°",Low,Use maps for relative patterns only; note in r...
5,DQ-06,ev vs geospatial,Unit mismatch: kWh vs MW,Demand is energy per event; capacity is power,High,Convert with an explicit charging-window assum...
6,DQ-07,ev_distribution,"Short history, small sample",97 events over ~5.5 weeks,Medium,Scenario-based growth instead of time-series f...


## 8. Summary and next step

| Question | Answer |
|---|---|
| What do we have? | 97 unique EV charging events across 49 substations (~5.5 weeks), 50 substation records, and 97 weather readings |
| Is it clean? | Complete (no nulls), but 81% of EV rows are duplicates, one substation key is duplicated, and weather labels are unreliable |
| Does it connect? | Yes. EV ↔ weather on Timestamp + Substation_ID, and EV ↔ substations on Substation_ID (after fixing Substation_11) |
| Biggest modelling risk | Converting kWh demand to MW load. This will be made explicit and stress-tested |

**Next:** `02_data_transformation.ipynb`: apply DQ-01 to DQ-04, engineer time features (hour, weekday, peak flag), encode the categorical fields, and build one analysis-ready table joining demand, weather, and substation capacity.